# Day 032 | Data Preprocessing Pipeline## 100 Days of ML for Geotechnical Engineering**Phase:** Data Science Essentials**Author:** Ripon Chandra Malo  **Date:** 2026---## What We Will Learn Today| Concept | What It Means | Geotechnical Example ||---------|--------------|---------------------|| **StandardScaler** | Zero mean, unit variance scaling | Normalize SPT N-values across different scales || **MinMaxScaler** | Scale to [0, 1] range | Normalize depth (1-30m) and N-value (1-50) to same range || **RobustScaler** | Uses median/IQR, resists outliers | Handle borehole data with refusal values (N>50) || **Train/Test Split** | Separate data for training and evaluation | Hold out 20% of samples for validation || **ColumnTransformer** | Different transforms for different columns | Scale numbers, encode soil_type, in one step || **Pipeline** | Chain preprocessing + model steps | Reusable preprocessing for any geotechnical dataset |---## Our Geotechnical ProblemWe need to prepare borehole data for an ML model that predicts soil properties. The challenge: our data has **mixed types** (numerical measurements + categorical soil types), **different scales** (depth in meters vs. unit weight in kN/m³), and potential **outliers** (refusal N-values).**Goal:** Build a reusable sklearn Pipeline that handles all preprocessing in one clean, reproducible step.---

## Part 1: Why Scaling MattersMany ML algorithms are sensitive to feature scales:- **Depth** ranges from 1 to 30 meters- **N_value** ranges from 1 to 50+- **Unit weight** ranges from 16 to 21 kN/m³Without scaling, the depth feature would dominate simply because its numbers are larger — not because it is more important.---

In [ ]:
# ─────────────────────────────────────────────────────# LOAD DATA AND EXAMINE SCALES# ─────────────────────────────────────────────────────import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScalerfrom sklearn.model_selection import train_test_splitfrom sklearn.compose import ColumnTransformerfrom sklearn.pipeline import Pipelinefrom sklearn.preprocessing import OneHotEncoderimport warningswarnings.filterwarnings('ignore')# ── Load the borehole dataset from Day 031 ──df = pd.read_csv("../Day_031/raw_soil_data.csv")print("Raw Data Overview")print("=" * 55)print(f"Shape: {df.shape}")print()# Show the scale differencesprint("Feature Ranges (showing scale mismatch):")print("─" * 55)for col in ['depth', 'N_value', 'water_content', 'unit_weight', 'fines_content']:    rng = df[col].max() - df[col].min()    print(f"  {col:20s}  min={df[col].min():8.1f}  max={df[col].max():8.1f}  range={rng:.1f}")print()print("Problem: depth range (~29) vs unit_weight range (~4)")print("ML algorithms would think depth is 7x more important!")

## Part 2: StandardScaler — The Most Common ChoiceStandardScaler transforms each feature to have **mean = 0** and **standard deviation = 1**.**Formula:** `z = (x - mean) / std`This works well when your data is approximately normally distributed.---

In [ ]:
# ─────────────────────────────────────────────────────# STANDARD SCALER# ─────────────────────────────────────────────────────numeric_features = ['depth', 'N_value', 'water_content', 'unit_weight', 'fines_content']X_numeric = df[numeric_features].copy()scaler_std = StandardScaler()X_standard = scaler_std.fit_transform(X_numeric)X_standard_df = pd.DataFrame(X_standard, columns=numeric_features)print("StandardScaler Results")print("=" * 55)print(f"{'Feature':20s} {'Mean':>10s} {'Std':>10s} {'Min':>10s} {'Max':>10s}")print("─" * 55)for col in numeric_features:    vals = X_standard_df[col]    print(f"{col:20s} {vals.mean():10.4f} {vals.std():10.4f} {vals.min():10.2f} {vals.max():10.2f}")print()print("All features now centered at 0 with std of 1.")print("Negative values = below average, positive = above average.")

## Part 3: MinMaxScaler — Bounded [0, 1] RangeMinMaxScaler scales each feature to a fixed range, typically [0, 1].**Formula:** `x_scaled = (x - x_min) / (x_max - x_min)`Best when you need bounded values (e.g., for neural networks) or when the data is not normally distributed.---

In [ ]:
# ─────────────────────────────────────────────────────# MINMAX SCALER# ─────────────────────────────────────────────────────scaler_mm = MinMaxScaler()X_minmax = scaler_mm.fit_transform(X_numeric)X_minmax_df = pd.DataFrame(X_minmax, columns=numeric_features)print("MinMaxScaler Results")print("=" * 55)print(f"{'Feature':20s} {'Min':>10s} {'Max':>10s} {'Mean':>10s}")print("─" * 55)for col in numeric_features:    vals = X_minmax_df[col]    print(f"{col:20s} {vals.min():10.4f} {vals.max():10.4f} {vals.mean():10.4f}")print()print("All features now in [0, 1] range.")print("0 = minimum observed value, 1 = maximum observed value.")

## Part 4: RobustScaler — Handling OutliersRobustScaler uses the **median** and **interquartile range (IQR)** instead of mean and standard deviation. This makes it resistant to outliers.**Formula:** `x_scaled = (x - median) / IQR`In geotechnical data, outliers are common: refusal N-values (>50), unusually high water content in organic soils, extreme unit weights.---

In [ ]:
# ─────────────────────────────────────────────────────# ROBUST SCALER# ─────────────────────────────────────────────────────scaler_rob = RobustScaler()X_robust = scaler_rob.fit_transform(X_numeric)X_robust_df = pd.DataFrame(X_robust, columns=numeric_features)print("RobustScaler Results")print("=" * 55)print(f"{'Feature':20s} {'Median':>10s} {'IQR':>10s} {'Min':>10s} {'Max':>10s}")print("─" * 55)for col in numeric_features:    vals = X_robust_df[col]    orig = X_numeric[col]    print(f"{col:20s} {orig.median():10.2f} {orig.quantile(0.75)-orig.quantile(0.25):10.2f} {vals.min():10.2f} {vals.max():10.2f}")print()print("RobustScaler: 0 = median, 1 = one IQR above median.")print("Less affected by extreme values than StandardScaler.")

In [ ]:
# ─────────────────────────────────────────────────────# VISUAL COMPARISON OF ALL THREE SCALERS# ─────────────────────────────────────────────────────fig, axes = plt.subplots(1, 3, figsize=(15, 4))feature = 'N_value'idx = numeric_features.index(feature)axes[0].hist(X_standard_df[feature], bins=20, color='steelblue', edgecolor='black', alpha=0.7)axes[0].set_title(f'StandardScaler\n(mean=0, std=1)', fontweight='bold')axes[0].set_xlabel(f'Scaled {feature}')axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5)axes[1].hist(X_minmax_df[feature], bins=20, color='coral', edgecolor='black', alpha=0.7)axes[1].set_title(f'MinMaxScaler\n(range [0,1])', fontweight='bold')axes[1].set_xlabel(f'Scaled {feature}')axes[2].hist(X_robust_df[feature], bins=20, color='mediumseagreen', edgecolor='black', alpha=0.7)axes[2].set_title(f'RobustScaler\n(median=0, IQR=1)', fontweight='bold')axes[2].set_xlabel(f'Scaled {feature}')axes[2].axvline(0, color='red', linestyle='--', linewidth=1.5)plt.suptitle(f'Scaling Comparison for {feature}', fontsize=13, fontweight='bold', y=1.02)plt.tight_layout()plt.savefig('scaler_comparison.png', dpi=100, bbox_inches='tight')plt.show()print("Which scaler to choose?")print("  StandardScaler : Normal-ish data, no extreme outliers")print("  MinMaxScaler   : Need bounded range, neural networks")print("  RobustScaler   : Data with outliers (refusal values, etc.)")

## Part 5: Train/Test Split — Never Peek at Test DataBefore preprocessing, we split our data into **training** and **test** sets. This prevents **data leakage** — the model must never see test data during training or preprocessing.**Critical rule:** Fit scalers on training data only, then transform both train and test.---

In [ ]:
# ─────────────────────────────────────────────────────# TRAIN/TEST SPLIT# ─────────────────────────────────────────────────────# ── Define features (X) and target (y) ──# Let's predict N_value from other featuresfeature_cols = ['depth', 'water_content', 'unit_weight', 'fines_content',                'liquid_limit', 'plasticity_index']target_col = 'N_value'X = df[feature_cols]y = df[target_col]# ── Split: 80% train, 20% test ──X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42)print("Train/Test Split")print("=" * 55)print(f"Total samples  : {len(df)}")print(f"Training set   : {len(X_train)} samples ({len(X_train)/len(df)*100:.0f}%)")print(f"Test set       : {len(X_test)} samples ({len(X_test)/len(df)*100:.0f}%)")print()# ── Correct approach: fit on TRAIN, transform both ──scaler = StandardScaler()scaler.fit(X_train)                    # Learn from training data ONLYX_train_scaled = scaler.transform(X_train)   # Transform training dataX_test_scaled = scaler.transform(X_test)     # Transform test data (same parameters!)print("Scaler learned from training set:")print(f"  Means: {dict(zip(feature_cols, scaler.mean_.round(2)))}")print()print("WRONG: scaler.fit(X_test)  ← NEVER fit on test data!")print("RIGHT: scaler.transform(X_test)  ← Only transform test data")

## Part 6: ColumnTransformer — Mixed Data TypesReal geotechnical datasets have both **numerical** features (depth, N-value) and **categorical** features (soil_type). They need different preprocessing:- Numerical: Scale with StandardScaler- Categorical: Encode with OneHotEncoder`ColumnTransformer` applies different transformers to different columns in one step.---

In [ ]:
# ─────────────────────────────────────────────────────# COLUMN TRANSFORMER FOR MIXED TYPES# ─────────────────────────────────────────────────────from sklearn.compose import ColumnTransformer# ── Define column groups ──num_features = ['depth', 'water_content', 'unit_weight', 'fines_content',                'liquid_limit', 'plasticity_index']cat_features = ['soil_type']# ── Build ColumnTransformer ──preprocessor = ColumnTransformer(    transformers=[        ('num', StandardScaler(), num_features),        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)    ],    remainder='drop'  # Drop columns not listed)# ── Prepare data with both types ──X_full = df[num_features + cat_features]y_full = df['N_value']X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(    X_full, y_full, test_size=0.2, random_state=42)# ── Fit and transform ──X_train_processed = preprocessor.fit_transform(X_train_f)X_test_processed = preprocessor.transform(X_test_f)# ── Get feature names ──ohe_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_features)all_names = list(num_features) + list(ohe_names)print("ColumnTransformer Results")print("=" * 55)print(f"Input features  : {len(num_features)} numerical + {len(cat_features)} categorical")print(f"Output features : {X_train_processed.shape[1]} total")print(f"Feature names   : {all_names}")print()print("First 3 processed training samples:")processed_df = pd.DataFrame(X_train_processed[:3], columns=all_names)print(processed_df.round(3).to_string())

## Part 7: sklearn Pipeline — Chain Everything TogetherA Pipeline chains preprocessing steps and (optionally) a model into one object. This ensures:1. **No data leakage** — fit/transform is handled correctly2. **Reproducibility** — same preprocessing every time3. **Clean code** — one object does everything---

In [ ]:
# ─────────────────────────────────────────────────────# BUILDING A COMPLETE PIPELINE# ─────────────────────────────────────────────────────from sklearn.pipeline import Pipelinefrom sklearn.linear_model import LinearRegression# ── Step 1: Define the preprocessor (same as before) ──preprocessor = ColumnTransformer(    transformers=[        ('num', StandardScaler(), num_features),        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)    ])# ── Step 2: Chain preprocessor + model into a Pipeline ──pipeline = Pipeline(steps=[    ('preprocessor', preprocessor),    ('model', LinearRegression())])print("Pipeline Structure:")print("=" * 55)for name, step in pipeline.steps:    print(f"  Step: {name:15s} → {type(step).__name__}")print()# ── Step 3: Fit the entire pipeline ──pipeline.fit(X_train_f, y_train_f)# ── Step 4: Predict (preprocessing happens automatically!) ──y_pred_train = pipeline.predict(X_train_f)y_pred_test = pipeline.predict(X_test_f)train_score = pipeline.score(X_train_f, y_train_f)test_score = pipeline.score(X_test_f, y_test_f)print(f"Pipeline Performance (R² score):")print(f"  Training : {train_score:.4f}")print(f"  Test     : {test_score:.4f}")print()print("The beauty: pipeline.predict(new_data) handles ALL")print("preprocessing internally. No manual scaling needed!")

In [ ]:
# ─────────────────────────────────────────────────────# TRYING DIFFERENT SCALERS IN THE PIPELINE# ─────────────────────────────────────────────────────# The Pipeline makes it easy to swap componentsresults = {}for scaler_name, scaler_obj in [('StandardScaler', StandardScaler()),                                  ('MinMaxScaler', MinMaxScaler()),                                  ('RobustScaler', RobustScaler())]:        pipe = Pipeline(steps=[        ('preprocessor', ColumnTransformer(            transformers=[                ('num', scaler_obj, num_features),                ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)            ]        )),        ('model', LinearRegression())    ])        pipe.fit(X_train_f, y_train_f)    train_r2 = pipe.score(X_train_f, y_train_f)    test_r2 = pipe.score(X_test_f, y_test_f)    results[scaler_name] = {'train': train_r2, 'test': test_r2}print("Scaler Comparison (R² score)")print("=" * 55)print(f"{'Scaler':20s} {'Train R²':>12s} {'Test R²':>12s}")print("─" * 55)for name, scores in results.items():    print(f"{name:20s} {scores['train']:12.4f} {scores['test']:12.4f}")print()print("For linear regression, all scalers give the same result!")print("The choice matters more for distance-based models (KNN, SVM).")

## Part 8: Complete Application — Reusable Preprocessing PipelineLet us build a production-quality preprocessing pipeline that can be saved and reused for any geotechnical dataset.---

In [ ]:
# ═════════════════════════════════════════════════════# REUSABLE GEOTECHNICAL PREPROCESSING PIPELINE# ═════════════════════════════════════════════════════from sklearn.pipeline import Pipelinefrom sklearn.compose import ColumnTransformerfrom sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformerimport pickle# ── Configuration ──NUM_FEATURES = ['depth', 'water_content', 'unit_weight',                 'fines_content', 'liquid_limit', 'plasticity_index']CAT_FEATURES = ['soil_type']# ── Build the pipeline ──geotech_pipeline = Pipeline(steps=[    ('preprocessor', ColumnTransformer(        transformers=[            ('num', Pipeline(steps=[                ('scaler', RobustScaler()),  # Robust to outliers            ]), NUM_FEATURES),            ('cat', OneHotEncoder(drop='first', sparse_output=False,                                   handle_unknown='ignore'), CAT_FEATURES)        ],        remainder='drop'    ))])# ── Fit on training data ──geotech_pipeline.fit(X_train_f)# ── Transform both sets ──X_train_ready = geotech_pipeline.transform(X_train_f)X_test_ready = geotech_pipeline.transform(X_test_f)print("Geotechnical Preprocessing Pipeline")print("=" * 55)print(f"Input  : {X_train_f.shape[1]} raw features (mixed types)")print(f"Output : {X_train_ready.shape[1]} preprocessed features (all numeric)")print(f"Train  : {X_train_ready.shape}")print(f"Test   : {X_test_ready.shape}")print()# ── Save the pipeline for reuse ──with open('geotech_pipeline.pkl', 'wb') as f:    pickle.dump(geotech_pipeline, f)print("Pipeline saved to geotech_pipeline.pkl")print("Load with: pipeline = pickle.load(open('geotech_pipeline.pkl', 'rb'))")print()print("Usage: X_new_processed = geotech_pipeline.transform(new_data)")

---## Practice Exercises### Exercise 1: Scale and compareApply all three scalers to the `water_content` column. Plot histograms of the scaled values side by side. Which scaler handles the distribution best?### Exercise 2: Pipeline with feature engineeringCreate a pipeline that first adds `depth²` as a feature (using `FunctionTransformer`), then scales all numerical features.### Exercise 3: Stratified splitUse `train_test_split` with `stratify=df['soil_type']` to ensure each soil type is equally represented in train and test. Check the distribution.---

In [ ]:
# ─────────────────────────────────────────────────────# EXERCISE 1: Scale water_content with all three scalers# ─────────────────────────────────────────────────────# wc = df[['water_content']]# fig, axes = plt.subplots(1, 3, figsize=(15, 4))# for ax, (name, scaler) in zip(axes, [('Standard', StandardScaler()),#                                        ('MinMax', MinMaxScaler()),#                                        ('Robust', RobustScaler())]):#     scaled = scaler.fit_transform(wc)#     ax.hist(scaled, bins=20, edgecolor='black', alpha=0.7)#     ax.set_title(name)# plt.tight_layout()# plt.show()

In [ ]:
# ─────────────────────────────────────────────────────# EXERCISE 2: Pipeline with depth²# ─────────────────────────────────────────────────────# def add_depth_squared(X):#     X = X.copy()#     X['depth_sq'] = X['depth'] ** 2#     return X## from sklearn.preprocessing import FunctionTransformer# feat_eng = FunctionTransformer(add_depth_squared)# pipe = Pipeline([('features', feat_eng), ('scaler', StandardScaler())])

In [ ]:
# ─────────────────────────────────────────────────────# EXERCISE 3: Stratified split# ─────────────────────────────────────────────────────# X_tr, X_te, y_tr, y_te = train_test_split(#     X_full, y_full, test_size=0.2, random_state=42,#     stratify=df['soil_type']# )# print("Training soil type distribution:")# print(X_tr['soil_type'].value_counts(normalize=True).round(3))

---## Key Takeaways1. **Always scale numerical features** before using distance-based or gradient-based ML algorithms. The choice of scaler depends on your data distribution and outlier situation.2. **RobustScaler is often best for geotechnical data** because field measurements frequently contain outliers (refusal values, equipment errors, unusual soil conditions).3. **Never fit preprocessors on test data.** Fit on training data, then transform both. This prevents data leakage.4. **ColumnTransformer handles mixed types** — apply StandardScaler to numbers and OneHotEncoder to categories in one step.5. **Pipeline is your best friend.** It chains preprocessing and modeling into one reproducible, leak-proof object. Save it with pickle for deployment.---## How This Connects to ML| What We Did Today | What It Becomes in ML ||---|---|| StandardScaler | Required for SVM, KNN, neural networks || MinMaxScaler | Required for neural networks (bounded activations) || RobustScaler | Best for real-world geotechnical data with outliers || Train/test split | Foundation of model evaluation || ColumnTransformer | Production-grade mixed-type preprocessing || Pipeline | Deployment-ready preprocessing + prediction |---*Day 032 of 100 — Keep going!*